In [1]:

## What your full structure now looks like

#ai-learning/
#├── ai-base/                    ← venv, don't touch
#├── requirements/
#│   ├── base.txt
#│   ├── general_ai.txt
#│   ├── llm.txt
#│   └── docling.txt
#├── notebooks/
#│   ├── general_ai.ipynb
#│   ├── llm.ipynb
#│   └── docling.ipynb
#├── models/
#│   ├── general_ai/             ← e.g. sklearn, custom torch models
#│   ├── llm/                    ← e.g. llama, mistral, phi weights
#│   └── docling/                ← e.g. OCR, layout detection models
#└── datasets/
#    ├── raw/                    ← original files, never modify these
#    └── processed/              ← cleaned, chunked, ready to use
    ## inside llama
    #models/
    #├── llm/
    #│   └── tinyllama/
    #│       ├── config.json
    #│       ├── tokenizer.json
    #│       ├── tokenizer_config.json
    #│       └── model.safetensors     ← the weights (~1.1GB)
    #├── general_ai/
    #└── docling/

In [ ]:
# ── Cell 0: check resources ──────────────────────────────────────────
import psutil
ram = psutil.virtual_memory()
print(f"Total RAM   : {ram.total / 1e9:.1f} GB")
print(f"Available   : {ram.available / 1e9:.1f} GB")

if ram.available < 3e9:
    print("Warning: less than 3GB free — model may crash. Close other apps first.")
else:
    print("RAM looks good.")

# You need at least 3GB free before loading the model

In [ ]:
# ── Cell 1: import libraries ─────────────────────────────────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
print(f"imported")

In [ ]:
# ── Cell 2: detect device ─────────────────────────────────────────────

print("Imports OK")

if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.float16
    print(f"GPU found    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float16
    print("Apple Silicon GPU (MPS) found")
else:
    device = "cpu"
    dtype = torch.float32   # float16 is unstable on CPU
    print("No GPU — running on CPU (will be slow)")

print(f"\nUsing device : {device}")
print(f"Using dtype  : {dtype}")

In [ ]:
# ── Cell 3: load model ────────────────────────────────────────────────
model_path = "../models/llm/tinyllama"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype=dtype,       # ← was wrongly written as dtype= before
    device_map="auto"        # auto picks cuda > mps > cpu
)

print(f"Model loaded on : {device}")
print(f"Memory used     : {psutil.virtual_memory().used / 1e9:.1f} GB")

In [ ]:
# ── Cell 4: run a prompt ──────────────────────────────────────────────
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user",   "content": "Explain what a neural network is in 2 sentences."}
]

input_ids = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt",
    add_generation_prompt=True   # tells the model a response is expected
).to(model.device)

with torch.no_grad():            # saves memory — no gradients needed for inference
    output = model.generate(
        input_ids,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        repetition_penalty=1.1   # reduces repetitive output, common with small models
    )

# decode only the new tokens (strip the prompt from the output)
response = tokenizer.decode(
    output[0][input_ids.shape[-1]:],   # ← slices off the input, shows only the reply
    skip_special_tokens=True
)

print("── Response ──────────────────────────────")
print(response)

In [ ]:
# ── Cell 5: memory report after inference ────────────────────────────
print(f"RAM used after inference : {psutil.virtual_memory().used / 1e9:.1f} GB")

if device == "cuda":
    print(f"VRAM used               : {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"VRAM reserved           : {torch.cuda.memory_reserved() / 1e9:.2f} GB")



In [ ]:
# ── Cell 6: interactive prompt loop ──────────────────────────────────
# Run this cell and keep chatting — type 'quit' to stop

conversation = [
    {"role": "system", "content": "You are a helpful assistant."}
]

print("TinyLlama is ready. Type your message below (or 'quit' to exit).\n")

while True:
    user_input = input("You: ").strip()
    
    if user_input.lower() in ["quit", "exit", "q"]:
        print("Ending conversation.")
        break
    if not user_input:
        continue

    conversation.append({"role": "user", "content": user_input})

    input_ids = tokenizer.apply_chat_template(
        conversation,
        return_tensors="pt",
        add_generation_prompt=True
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            repetition_penalty=1.1
        )

    reply = tokenizer.decode(
        output[0][input_ids.shape[-1]:],
        skip_special_tokens=True
    ).strip()

    conversation.append({"role": "assistant", "content": reply})
    print(f"\nTinyLlama: {reply}\n")

In [ ]:
# ── Cell 7: save a conversation to disk ──────────────────────────────
# Run after Cell 6 to save the chat history as a JSON file

import json
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_path = f"../datasets/processed/chat_{timestamp}.json"

with open(save_path, "w") as f:
    json.dump(conversation, f, indent=2)

print(f"Conversation saved to: {save_path}")

In [ ]:
# ── Cell 8: reload a saved conversation ──────────────────────────────
# Use this to continue a previous chat session

import json
import glob

# List all saved conversations
saved = sorted(glob.glob("../datasets/processed/chat_*.json"))
print("Saved conversations found:")
for i, path in enumerate(saved):
    print(f"  [{i}] {path}")

# Load the most recent one (change index to pick a different one)
with open(saved[-1]) as f:
    conversation = json.load(f)

print(f"\nLoaded {len(conversation)} messages from {saved[-1]}")
print("Last exchange:")
for msg in conversation[-2:]:
    print(f"  {msg['role'].upper()}: {msg['content'][:80]}...")

In [ ]:
# ── Cell 9: clean up and free memory ─────────────────────────────────

import gc

# safely delete only if they exist in memory
if 'model' in dir():
    del model
    print("Model unloaded.")
else:
    print("No model in memory — nothing to unload.")

if 'tokenizer' in dir():
    del tokenizer
    print("Tokenizer unloaded.")
else:
    print("No tokenizer in memory — nothing to unload.")

gc.collect()

if 'device' in dir() and device == "cuda":
    torch.cuda.empty_cache()
    print(f"VRAM after cleanup : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

ram_after = psutil.virtual_memory()
print(f"RAM after cleanup  : {ram_after.used / 1e9:.1f} GB used")
print("Done.")